# Lab 6 – PySpark DataFrames: Przetwarzanie danych ustrukturyzowanych

W poprzednim laboratorium poznaliśmy niskopoziomowy interfejs **RDD (Resilient Distributed Datasets)**. Choć RDD daje pełną kontrolę nad rozproszonymi kolekcjami obiektów Pythona, to przy pracy z danymi tabelarycznymi ma istotne ograniczenia:
1. **Brak informacji o schemacie danych:** Silnik Spark traktuje elementy RDD jako nieprzezroczyste obiekty Pythona (tzw. *black-box*), które muszą być serializowane (za pomocą `pickle`) pomiędzy maszyną wirtualną Javy (JVM) a procesami roboczymi Pythona.
2. **Brak automatycznej optymalizacji zapytań:** Kolejność operacji `map` i `filter` w RDD zależy wyłącznie od programisty.

W tym laboratorium przechodzimy do wysokopoziomowego API **Spark DataFrame (`pyspark.sql`)**, które jest dziś podstawowym narzędziem pracy inżynierów danych i analityków Big Data.

---

### Dlaczego Spark DataFrame jest znacznie szybszy niż RDD?
* **Optymalizator zapytań Catalyst (*Catalyst Optimizer*):** Każda operacja na obiekcie `DataFrame` buduje logiczny plan zapytania, który przed uruchomieniem jest automatycznie optymalizowany (np. *Predicate Pushdown* – wypchnięcie filtrów bezpośrednio do warstwy odczytu plików, *Column Pruning* – wczytywanie z dysku wyłącznie używanych kolumn).
* **Silnik wykonawczy Tungsten (*Project Tungsten*):** Dane w `DataFrame` są przechowywane w wewnętrznym, binarnym formacie kolumnowym bezpośrednio w pamięci JVM (oraz poza stertą – *off-heap*), co eliminuje narzut serializacji Python $\leftrightarrow$ JVM i pozwala na generowanie natywnego bajtkodu (*Whole-Stage Code Generation*).
* **DataFrame a Dataset:** W ekosystemie Spark istnieje również pojęcie `Dataset[T]` – silnie typowanej kolekcji dostępnej w językach kompilowanych (**Scala** oraz **Java**), gdzie typy sprawdzane są na etapie kompilacji. W języku Python, ze względu na jego dynamiczną naturę, operujemy bezpośrednio na ujednoliconym interfejsie `DataFrame` (który pod spodem odpowiada `Dataset[Row]`).

> **Dokumentacja PySpark SQL & DataFrames:**
> * [PySpark DataFrame API Reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)
> * [Wbudowane funkcje `pyspark.sql.functions`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
> * [Typy danych `pyspark.sql.types`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html)

## Moduł 0: Uruchomienie sesji `SparkSession`

Podobnie jak w poprzednim laboratorium, dynamicznie przypisujemy ścieżkę bieżącego interpretera Pythona (`sys.executable`), dzięki czemu notatnik działa bez modyfikacji zarówno w lokalnym środowisku wirtualnym (Windows / Linux / macOS), jak i w kontenerze Docker.

In [80]:
import os
import sys

# Wskazanie bieżącego interpretera Pythona dla procesów Drivera i Executorów:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Interpreter Pythona:", sys.executable)
print("Wersja Pythona:", sys.version.split()[0])

Interpreter Pythona: c:\Analiza-duzych-zbiorow-danych\.venv_spark\Scripts\python.exe
Wersja Pythona: 3.11.9


In [81]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    DecimalType,
)

# Na Windowsie zapis Parquet ustawia uprawnienia plików przez winutils.exe.
# Hadoop czyta HADOOP_HOME tylko raz, przy starcie JVM.
if os.name == "nt":
    hadoop_home = os.path.abspath(os.path.join(os.getcwd(), "hadoop"))
    bin_dir = os.path.join(hadoop_home, "bin")
    os.makedirs(bin_dir, exist_ok=True)
    winutils_path = os.path.join(bin_dir, "winutils.exe")
    hadoop_dll_path = os.path.join(bin_dir, "hadoop.dll")
    if not (os.path.isfile(winutils_path) and os.path.isfile(hadoop_dll_path)):
        import urllib.request

        winutils_url = (
            "https://raw.githubusercontent.com/cdarlint/winutils/master/"
            "hadoop-3.3.6/bin/"
        )
        print("Pobieranie winutils.exe (wymagane do zapisu Parquet na Windows)...")
        urllib.request.urlretrieve(winutils_url + "winutils.exe", winutils_path)
        urllib.request.urlretrieve(winutils_url + "hadoop.dll", hadoop_dll_path)
    os.environ["HADOOP_HOME"] = hadoop_home
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")
    print("HADOOP_HOME:", hadoop_home)

spark = (
    SparkSession.builder.master("local[2]")
    .appName("Lab06-PySpark-DataFrames")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .getOrCreate()
)

sc = spark.sparkContext
if os.name == "nt" and not spark._jvm.java.lang.System.getenv("HADOOP_HOME"):
    print(
        "Ta sesja Sparka wystartowała bez HADOOP_HOME. "
        "Zrestartuj kernel i uruchom komórki od początku, potem powtórz zapis Parquet."
    )
print("Wersja Apache Spark:", spark.version)
print("Panel diagnostyczny Spark Web UI: http://localhost:4040")
sc

Wersja Apache Spark: 4.1.4
Panel diagnostyczny Spark Web UI: http://localhost:4040


<SparkContext master=local[2] appName=Lab06-PySpark-DataFrames>

---
## Moduł 1: Pozyskiwanie danych rzeczywistych, pułapki CSV i schematy (`StructType`)

### Praktyczny przypadek użycia: Analiza recenzji kulinarnych (*UCI Recipe Reviews*)
W pierwszych czterech modułach będziemy pracować na rzeczywistym zbiorze danych **[Recipe Reviews and User Feedback Dataset](https://archive.ics.uci.edu/dataset/911/recipe+reviews+and+user+feedback+dataset)** z repozytorium UCI Machine Learning. Zbiór zawiera ponad 18 tysięcy recenzji przepisów kulinarnych wraz z informacjami o autorach, ich reputacji, liczbie polubień (`thumbs_up`, `thumbs_down`), odpowiedziach oraz pełnej treści komentarzy (`text`).

#### 1.1. Pobranie i rozpakowanie danych (wieloplatformowo w Pythonie)
Zamiast wywoływać nieprzenośne polecenia powłoki systemowej (`!wget`, `!ls`, `!head`), pobierzemy i rozpakujemy archiwum przy użyciu standardowej biblioteki Pythona (`urllib.request` oraz `zipfile`).

In [82]:
import urllib.request
import zipfile

DATA_DIR = "./data"
ZIP_PATH = os.path.join(DATA_DIR, "recipe_reviews.zip")
CSV_PATH = os.path.join(DATA_DIR, "Recipe Reviews and User Feedback Dataset.csv")
DATASET_URL = "https://archive.ics.uci.edu/static/public/911/recipe+reviews+and+user+feedback+dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(CSV_PATH):
    print("Pobieranie archiwum ze zbioru UCI...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print("Rozpakowywanie archiwum...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Gotowe!")
else:
    print("Plik CSV już istnieje na dysku:", CSV_PATH)

print("Zawartość katalogu ./data:", os.listdir(DATA_DIR))

Plik CSV już istnieje na dysku: ./data\Recipe Reviews and User Feedback Dataset.csv
Zawartość katalogu ./data: ['employee_parquet', 'Recipe Reviews and User Feedback Dataset.csv', 'recipe_reviews.zip']


In [83]:
# Podejrzyjmy pierwsze 3 linie surowego pliku CSV przy użyciu czystego Pythona:
with open(CSV_PATH, mode="r", encoding="utf-8") as f:
    for i in range(3):
        print(f"Linia {i}: {f.readline().strip()[:120]}...")

Linia 0: ,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thu...
Linia 1: 0,001,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM,u_9iFLIhMa8QaG,Jeri326,1,1665619889,0,0,0...
Linia 2: 1,001,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY,u_Lu6p25tmE77j,Mark467,50,1665277687,0,7,...


#### 1.2. Pułapki wczytywania plików CSV w Sparku
Domyślne wywołanie `spark.read.csv(CSV_PATH, header=True)` kryje w sobie **dwie typowe pułapki**:
1. **Brak wnioskowania typów (`inferSchema=False` domyślnie):** Wszystkie kolumny (nawet oceny `stars` czy liczniki `thumbs_up`) zostają wczytane jako tekst (`string`).
2. **Recenzje wieloliniowe i cudzysłowy wewnątrz tekstu (`multiLine=False` domyślnie):** Użytkownicy w treści recenzji (`text`) często używają znaków nowej linii (`\n`) oraz cudzysłowów (`"`). Bez parametrów `multiLine=True` oraz `escape='"'` Spark potraktuje złamanie linii wewnątrz komentarza jako nowy wiersz tabeli, co spowoduje przesunięcie kolumn i pojawienie się fałszywych wartości `NULL`!

Sprawdźmy to w praktyce:

In [84]:
# 1. Wczytanie zupełnie domyślne (wszystkie kolumny jako string, brak multiLine):
df_all_strings = spark.read.csv(CSV_PATH, header=True, sep=",")
print("Schemat domyślny (inferSchema=False):")
df_all_strings.printSchema()

Schemat domyślny (inferSchema=False):
root
 |-- _c0: string (nullable = true)
 |-- recipe_number: string (nullable = true)
 |-- recipe_code: string (nullable = true)
 |-- recipe_name: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_reputation: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- reply_count: string (nullable = true)
 |-- thumbs_up: string (nullable = true)
 |-- thumbs_down: string (nullable = true)
 |-- stars: string (nullable = true)
 |-- best_score: string (nullable = true)
 |-- text: string (nullable = true)



In [85]:
# 2. Wczytanie z inferSchema=True, ale wciąż BEZ obsługi recenzji wieloliniowych (multiLine=False):
df_naive = spark.read.csv(CSV_PATH, header=True, sep=",", inferSchema=True)
print("Liczba wierszy (wczytanie naiwne bez multiLine):", df_naive.count())
df_naive.printSchema()

Liczba wierszy (wczytanie naiwne bez multiLine): 18268
root
 |-- _c0: string (nullable = true)
 |-- recipe_number: string (nullable = true)
 |-- recipe_code: string (nullable = true)
 |-- recipe_name: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_reputation: string (nullable = true)
 |-- created_at: integer (nullable = true)
 |-- reply_count: integer (nullable = true)
 |-- thumbs_up: integer (nullable = true)
 |-- thumbs_down: integer (nullable = true)
 |-- stars: integer (nullable = true)
 |-- best_score: integer (nullable = true)
 |-- text: string (nullable = true)



In [86]:
# 3. Wczytanie poprawne (inferSchema=True + multiLine=True + escape='"'):
df_reviews = spark.read.csv(
    CSV_PATH,
    header=True,
    sep=",",
    inferSchema=True,
    multiLine=True,
    escape='"',
)

print("Liczba wierszy (wczytanie poprawne z multiLine=True):", df_reviews.count())
df_reviews.printSchema()

Liczba wierszy (wczytanie poprawne z multiLine=True): 18182
root
 |-- _c0: integer (nullable = true)
 |-- recipe_number: integer (nullable = true)
 |-- recipe_code: integer (nullable = true)
 |-- recipe_name: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_reputation: integer (nullable = true)
 |-- created_at: integer (nullable = true)
 |-- reply_count: integer (nullable = true)
 |-- thumbs_up: integer (nullable = true)
 |-- thumbs_down: integer (nullable = true)
 |-- stars: integer (nullable = true)
 |-- best_score: integer (nullable = true)
 |-- text: string (nullable = true)



In [87]:
# Wyświetlenie pierwszych 5 wierszy (truncate=35 przycina długie teksty recenzji dla czytelności tabeli):
df_reviews.show(5, truncate=35)

+---+-------------+-----------+------------------+-----------------------------------+--------------+----------+---------------+----------+-----------+---------+-----------+-----+----------+-----------------------------------+
|_c0|recipe_number|recipe_code|       recipe_name|                         comment_id|       user_id| user_name|user_reputation|created_at|reply_count|thumbs_up|thumbs_down|stars|best_score|                               text|
+---+-------------+-----------+------------------+-----------------------------------+--------------+----------+---------------+----------+-----------+---------+-----------+-----+----------+-----------------------------------+
|  0|            1|      14299|Creamy White Chili|sp_aUSaElGf_14299_c_2G3aneMRgRMZ...|u_9iFLIhMa8QaG|   Jeri326|              1|1665619889|          0|        0|          0|    5|       527|I tweaked it a little, removed o...|
|  1|            1|      14299|Creamy White Chili|sp_aUSaElGf_14299_c_2FsPC83HtzCs...|u_Lu6p

#### 1.3. Jawne definiowanie schematu (`StructType`) i rzutowanie typów (`cast`)
W systemach produkcyjnych włączanie `inferSchema=True` na plikach CSV o rozmiarze setek gigabajtów bywa kosztowne, ponieważ wymaga **dodatkowego przebiegu odczytu danych**, aby odgadnąć typy kolumn. Ponadto surowe dane często zawierają puste łańcuchy `""` lub błędne znaki, które uniemożliwiają bezpośrednią konwersję w locie.

W takich sytuacjach definiujemy schemat jawnie za pomocą obiektów `StructType` oraz `StructField`, lub wczytujemy wrażliwe kolumny jako `StringType()`, a następnie rzutujemy je metodą `.cast()`:

In [88]:
# Przykład jawnego schematu oraz bezpiecznego rzutowania typów (cast):
sample_employees = [
    ("James", "U001", "Smith", "36636", "M", "3000.50"),
    ("Michael", "U002", "Rose", "40288", "M", "4000.00"),
    ("Robert", "U003", "Williams", "42114", "M", "4250.75"),
    ("Maria", "U004", "Jones", "39192", "F", "3900.00"),
    ("Jen", "U005", "Brown", None, "F", "1000.00"),
]

explicit_schema = StructType(
    [
        StructField("firstname", StringType(), nullable=True),
        StructField("user_id", StringType(), nullable=True),
        StructField("lastname", StringType(), nullable=True),
        StructField("emp_id", StringType(), nullable=True),
        StructField("gender", StringType(), nullable=True),
        StructField("salary", StringType(), nullable=True),
    ]
)

df_sample = spark.createDataFrame(data=sample_employees, schema=explicit_schema)

# Konwersja kolumn tekstowych na docelowe typy numeryczne po wczytaniu:
df_sample = df_sample.withColumn("emp_id", F.col("emp_id").cast(LongType())).withColumn(
    "salary", F.col("salary").cast(DecimalType(10, 2))
)

df_sample.printSchema()
df_sample.show()

root
 |-- firstname: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)

+---------+-------+--------+------+------+-------+
|firstname|user_id|lastname|emp_id|gender| salary|
+---------+-------+--------+------+------+-------+
|    James|   U001|   Smith| 36636|     M|3000.50|
|  Michael|   U002|    Rose| 40288|     M|4000.00|
|   Robert|   U003|Williams| 42114|     M|4250.75|
|    Maria|   U004|   Jones| 39192|     F|3900.00|
|      Jen|   U005|   Brown|  NULL|     F|1000.00|
+---------+-------+--------+------+------+-------+



### Ćwiczenie utrwalające 1 (Wczytywanie i porządkowanie schematu)
Pierwsza kolumna w zbiorze `df_reviews` nie miała nazwy w nagłówku pliku CSV, dlatego Spark nadał jej automatyczną nazwę `_c0`.

1. Korzystając z metody [`.withColumnRenamed(existing, new)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnRenamed.html), zmień w ramce `df_reviews` nazwę kolumny `_c0` na `review_row_id` (i przypisz wynik z powrotem do zmiennej `df_reviews`).
2. Sprawdź i wypisz różnicę w liczbie wierszy między błędnie wczytaną ramką `df_naive` a poprawnie wczytaną ramką `df_reviews` – ile nadmiarowych ("połamanych") wierszy powstało przez brak parametru `multiLine=True`?
3. Wyświetl 5 pierwszych wierszy zawierających wyłącznie kolumny: `review_row_id`, `recipe_name`, `user_name` oraz `stars`.

In [89]:
# Miejsce na rozwiązanie Ćwiczenia 1:

# 1. Zmiana nazwy kolumny _c0 na review_row_id
df_reviews = df_reviews.withColumnRenamed("_c0", "review_row_id")

# 2. Nadmiarowe ("połamane") wiersze przez brak multiLine=True
broken_rows = df_naive.count() - df_reviews.count()
print("Liczba wierszy df_naive:", df_naive.count())
print("Liczba wierszy df_reviews:", df_reviews.count())
print("Nadmiarowe (połamane) wiersze:", broken_rows)

# 3. Pierwsze 5 wierszy wybranych kolumn
df_reviews.select(
    "review_row_id", "recipe_name", "user_name", "stars"
).show(5)

Liczba wierszy df_naive: 18268
Liczba wierszy df_reviews: 18182
Nadmiarowe (połamane) wiersze: 86
+-------------+------------------+----------+-----+
|review_row_id|       recipe_name| user_name|stars|
+-------------+------------------+----------+-----+
|            0|Creamy White Chili|   Jeri326|    5|
|            1|Creamy White Chili|   Mark467|    5|
|            2|Creamy White Chili|Barbara566|    5|
|            3|Creamy White Chili|jeansch123|    0|
|            4|Creamy White Chili|  camper77|    0|
+-------------+------------------+----------+-----+
only showing top 5 rows


---
## Moduł 2: Wyrażenia kolumnowe (`Column API`), leniwa ewaluacja i transformacje

### 2.1. Odwołania do kolumn i projekcja (`select`)
Każda kolumna w `DataFrame` jest reprezentowana przez obiekt klasy [`pyspark.sql.Column`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html). Do kolumny możemy odwołać się na trzy równoważne sposoby:
* `df.user_name` (wygodne, ale nie działa gdy nazwa kolumny zawiera spacje lub pokrywa się z metodą DataFrame),
* `df["user_name"]` (styl znany z biblioteki `pandas`),
* `F.col("user_name")` (rekomendowany standard w PySparku – czytelny i niezależny od nazwy zmiennej przechowującej ramkę).

Podobnie jak w RDD, wywołanie `.select(...)` czy `.withColumn(...)` jest **transformacją leniwą (*lazy evaluation*)** – zwraca jedynie nowy obiekt `DataFrame` z zaktualizowanym planem zapytania, a obliczenia ruszają dopiero przy wywołaniu akcji (np. `.show()`, `.count()`, `.collect()`, `.write`).

In [90]:
# Upewnijmy się, że kolumna _c0 ma czytelny alias review_row_id również w dalszej części notatnika:
if "_c0" in df_reviews.columns:
    df_reviews = df_reviews.withColumnRenamed("_c0", "review_row_id")

# Trzy sposoby wskazania kolumny zwracają obiekt typu Column (bez pobierania danych!):
print("Typ obiektu kolumny:", type(df_reviews["user_name"]))
print("Leniwa transformacja select:", df_reviews.select("user_name", "user_reputation"))

# Dopiero akcja .show() uruchamia zadanie Spark:
df_reviews.select(
    F.col("user_name"),
    F.col("user_reputation"),
    F.col("stars"),
).show(5)

Typ obiektu kolumny: <class 'pyspark.sql.classic.column.Column'>
Leniwa transformacja select: DataFrame[user_name: string, user_reputation: int]
+----------+---------------+-----+
| user_name|user_reputation|stars|
+----------+---------------+-----+
|   Jeri326|              1|    5|
|   Mark467|             50|    5|
|Barbara566|             10|    5|
|jeansch123|              1|    0|
|  camper77|             10|    0|
+----------+---------------+-----+
only showing top 5 rows


> 💡 **Wskazówka (Eager Evaluation w Jupyterze):** W małych zbiorach testowych można włączyć automatyczne renderowanie ramek HTML w komórkach Jupytera za pomocą:
> `spark.conf.set("spark.sql.repl.eagerEval.enabled", True)`
> W pracy z dużymi zbiorami danych pozostawiamy jednak domyślne `False`, aby przypadkowe wypisanie zmiennej nie uruchamiało kosztownych zapytań.

### 2.2. Tworzenie nowych kolumn (`withColumn`), wyrażenia warunkowe (`when`/`otherwise`) i potoki `transform()`
Do wyliczania nowych cech wykorzystujemy wektorowe funkcje wbudowane z modułu `pyspark.sql.functions` (importowanego tradycyjnie jako `F`), które wykonują się natywnie wewnątrz maszyny JVM (bez kosztownego przesyłania wierszy do interpretera Pythona!).

Zobaczmy praktyczny przykład wzbogacenia recenzji o:
1. Datę utworzenia komentarza w czytelnym formacie (`F.from_unixtime`),
2. Kategorię sentymentu na podstawie liczby gwiazdek (`F.when(...).when(...).otherwise(...)`),
3. Funkcję pomocniczą przekazywaną do metody [`DataFrame.transform()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.transform.html), pozwalającą czysto łączyć kolejne kroki przetwarzania.

In [91]:
def enrich_reviews(df):
    return (
        df.withColumn(
            "created_date",
            F.to_timestamp(F.from_unixtime(F.col("created_at"))),
        )
        .withColumn(
            "sentiment_label",
            F.when(F.col("stars") >= 4, "Pozytywna (4-5*)")
            .when(F.col("stars") == 3, "Neutralna (3*)")
            .when(F.col("stars") >= 1, "Negatywna (1-2*)")
            .otherwise("Brak oceny (0*)"),
        )
        .withColumn("recipe_name_upper", F.upper(F.col("recipe_name")))
    )


df_reviews_enriched = df_reviews.transform(enrich_reviews)

df_reviews_enriched.select(
    "recipe_name_upper",
    "created_date",
    "stars",
    "sentiment_label",
).show(8, truncate=30)

+------------------+-------------------+-----+----------------+
| recipe_name_upper|       created_date|stars| sentiment_label|
+------------------+-------------------+-----+----------------+
|CREAMY WHITE CHILI|2022-10-13 02:11:29|    5|Pozytywna (4-5*)|
|CREAMY WHITE CHILI|2022-10-09 03:08:07|    5|Pozytywna (4-5*)|
|CREAMY WHITE CHILI|2022-09-29 00:35:57|    5|Pozytywna (4-5*)|
|CREAMY WHITE CHILI|2022-08-29 17:43:28|    0| Brak oceny (0*)|
|CREAMY WHITE CHILI|2022-10-04 22:03:43|    0| Brak oceny (0*)|
|CREAMY WHITE CHILI|2022-08-24 17:19:11|    5|Pozytywna (4-5*)|
|CREAMY WHITE CHILI|2022-02-05 20:20:05|    5|Pozytywna (4-5*)|
|CREAMY WHITE CHILI|2022-02-04 03:35:14|    0| Brak oceny (0*)|
+------------------+-------------------+-----+----------------+
only showing top 8 rows


### Ćwiczenie utrwalające 2 (Wyrażenia kolumnowe i nowe cechy)
Pracując na ramce `df_reviews_enriched`, utwórz dwie nowe kolumny za pomocą metody `.withColumn()`:
1. `net_thumbs` – bilans głosów pod recenzją, obliczany jako różnica: `thumbs_up - thumbs_down`,
2. `review_char_len` – długość treści recenzji w znakach, wyznaczona za pomocą funkcji [`F.length(F.col("text"))`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.length.html).

Następnie wyświetl 10 recenzji o największej długości tekstu (`review_char_len`), pokazując kolumny: `recipe_name`, `user_name`, `stars`, `net_thumbs` oraz `review_char_len` (do posortowania malejącego użyj `.orderBy(F.desc("review_char_len"))`).

In [92]:
# Miejsce na rozwiązanie Ćwiczenia 2:

# 1. Bilans głosów i 2. długość treści recenzji
df_reviews_enriched = (
    df_reviews_enriched.withColumn(
        "net_thumbs", F.col("thumbs_up") - F.col("thumbs_down")
    ).withColumn("review_char_len", F.length(F.col("text")))
)

# 10 najdłuższych recenzji
df_reviews_enriched.select(
    "recipe_name", "user_name", "stars", "net_thumbs", "review_char_len"
).orderBy(F.desc("review_char_len")).show(10)

+--------------------+-------------+-----+----------+---------------+
|         recipe_name|    user_name|stars|net_thumbs|review_char_len|
+--------------------+-------------+-----+----------+---------------+
|  Caramel Heavenlies| andreasmom2u|    5|         0|           2742|
|Favorite Chicken ...|       Regi M|    5|         2|           2548|
|Blueberry French ...| delowenstein|    5|         1|           2322|
|Amish Breakfast C...|       Regi M|    5|        -3|           2232|
|Peanut Butter Cho...|   Psalm127-3|    4|        -1|           2065|
|Homemade Peanut B...|     HeatherB|    5|        -3|           1757|
|     Seafood Lasagna|   TaraKuhafa|    5|         0|           1721|
|Enchilada Casser-...|    Cubbybear|    0|         0|           1679|
|   Cheeseburger Soup|   TanteCindy|    3|        -1|           1602|
|Basic Banana Muffins|Andromeda1971|    5|         0|           1559|
+--------------------+-------------+-----+----------+---------------+
only showing top 10 

---
## Moduł 3: Diagnostyka jakości danych (`NULL` / `NaN`), czyszczenie i filtrowanie

### 3.1. Zliczanie braków danych i diagnoza uszkodzonych wierszy
W PySparku rozróżniamy dwa rodzaje wartości brakujących:
* **`NULL` (`None` w Pythonie):** standardowy brak wartości w SQL, może wystąpić w kolumnie dowolnego typu (`isNull()`),
* **`NaN` (*Not a Number*):** specjalna wartość zmiennoprzecinkowa standardu IEEE 754, występująca wyłącznie w kolumnach numerycznych (`FloatType`, `DoubleType`) – sprawdzana funkcją `F.isnan()`.

Porównajmy teraz liczbę wartości `NULL` w ramce wczytanej **bez `multiLine=True` (`df_naive`)** oraz w ramce wczytanej **poprawnie (`df_reviews`)**:

In [93]:
def count_nulls(df):
    float_cols = {
        f.name for f in df.schema.fields if isinstance(f.dataType, DoubleType)
    }
    return df.select(
        [
            F.count(
                F.when(
                    (F.col(c).isNull() | F.isnan(c))
                    if c in float_cols
                    else F.col(c).isNull(),
                    c,
                )
            ).alias(c)
            for c in df.columns
        ]
    )


print("1. Liczba wartości NULL w ramce df_naive (wczytanej BEZ multiLine=True):")
count_nulls(df_naive).show()

print("2. Liczba wartości NULL w ramce df_reviews (wczytanej Z multiLine=True):")
count_nulls(df_reviews).show()

1. Liczba wartości NULL w ramce df_naive (wczytanej BEZ multiLine=True):
+---+-------------+-----------+-----------+----------+-------+---------+---------------+----------+-----------+---------+-----------+-----+----------+----+
|_c0|recipe_number|recipe_code|recipe_name|comment_id|user_id|user_name|user_reputation|created_at|reply_count|thumbs_up|thumbs_down|stars|best_score|text|
+---+-------------+-----------+-----------+----------+-------+---------+---------------+----------+-----------+---------+-----------+-----+----------+----+
|  0|           45|         64|         74|        77|     80|       83|             84|        86|         86|       86|         86|   86|        86|  86|
+---+-------------+-----------+-----------+----------+-------+---------+---------------+----------+-----------+---------+-----------+-----+----------+----+

2. Liczba wartości NULL w ramce df_reviews (wczytanej Z multiLine=True):
+-------------+-------------+-----------+-----------+----------+-------+-

In [94]:
# Zobaczmy, skąd wzięły się wartości NULL w kolumnie recipe_code w naiwnie wczytanej ramce df_naive:
# Pierwsza kolumna (_c0) zawiera dalsze części zdań z komentarzy złamanych znakiem nowej linii!
df_naive.filter(F.col("recipe_code").isNull()).select(
    "_c0", "recipe_number", "recipe_code", "recipe_name", "stars"
).show(8, truncate=45)

+---------------------------------------------+-------------+-----------+-----------+-----+
|                                          _c0|recipe_number|recipe_code|recipe_name|stars|
+---------------------------------------------+-------------+-----------+-----------+-----+
|                               Thank you!!!!"|         NULL|       NULL|       NULL| NULL|
|                          It was excellent!  |         NULL|       NULL|       NULL| NULL|
|The recipe was a bit high in sodium so I c...|         NULL|       NULL|       NULL| NULL|
|A shoutout to the wife of James569 with th...|         NULL|       NULL|       NULL| NULL|
|This recipe is destined for my next potluck."|         NULL|       NULL|       NULL| NULL|
| like the time it tales to cook. I;m disab...|         NULL|       NULL|       NULL| NULL|
|My one complaint is the recipe reads makes...|         NULL|       NULL|       NULL| NULL|
|Have never eaten this sort of hot dish bef...|         NULL|       NULL|       

In [95]:
# Nawet przy poprawnym wczytaniu warto zabezpieczyć potok przed ewentualnymi brakami danych lub pustymi recenzjami:
df_reviews_clean = df_reviews_enriched.na.drop().filter(F.length(F.trim(F.col("text"))) > 0)
print(f"Liczba wierszy w df_reviews_clean: {df_reviews_clean.count()}")

Liczba wierszy w df_reviews_clean: 18182


### 3.2. Filtrowanie wierszy (`filter` / `where`)
Metody `.filter()` oraz `.where()` są w PySparku dokładnymi synonimami. Warunki logiczne budujemy przy użyciu:
* operatorów porównania: `==`, `!=`, `>`, `>=`, `<`, `<=`,
* operatorów logicznych (pamiętając o **nawiasach wokół każdego warunku!**): `&` (AND), `|` (OR), `~` (NOT),
* metod tekstowych i zbiorowych klasy `Column`: `.startswith()`, `.endswith()`, `.contains()`, `.rlike()`, `.isin()`, `.between()`.

In [96]:
# Przykład 1: Użytkownicy, których nazwa zaczyna się na literę 'a' (bez względu na wielkość liter):
df_reviews_clean.filter(F.lower(F.col("user_name")).startswith("a")).select(
    "user_name", "user_reputation", "recipe_name", "stars"
).show(5, truncate=30)

# Przykład 2: Filtrowanie wielowarunkowe – recenzje 5-gwiazdkowe z co najmniej 10 polubieniami:
df_reviews_clean.filter(
    (F.col("stars") == 5) & (F.col("thumbs_up") >= 10)
).select("recipe_name", "user_name", "thumbs_up", "thumbs_down", "stars").show(
    5, truncate=30
)

+--------------+---------------+------------------+-----+
|     user_name|user_reputation|       recipe_name|stars|
+--------------+---------------+------------------+-----+
|         ahmom|              1|Creamy White Chili|    5|
|  annamossburg|              1|Creamy White Chili|    5|
|AshleyMcCauley|              1|Creamy White Chili|    5|
|          AAME|              1|Creamy White Chili|    0|
| AllisonJustus|              1|Creamy White Chili|    4|
+--------------+---------------+------------------+-----+
only showing top 5 rows
+------------------+--------------+---------+-----------+-----+
|       recipe_name|     user_name|thumbs_up|thumbs_down|stars|
+------------------+--------------+---------+-----------+-----+
|Creamy White Chili|     Sandy1256|       11|          0|    5|
|Creamy White Chili|      Scott967|       11|          1|    5|
|Creamy White Chili|     Kimosaabe|       15|          1|    5|
|Creamy White Chili|     OliveCorn|       16|          5|    5|
|Cream

### Ćwiczenie utrwalające 3 (Filtrowanie wielowarunkowe)
Korzystając z oczyszczonej ramki `df_reviews_clean`:
1. Przefiltruj recenzje, które spełniają **jednocześnie** trzy warunki:
   * ocena `stars` jest niska (1 lub 2 gwiazdki, np. `F.col("stars").isin(1, 2)` lub `F.col("stars").between(1, 2)`),
   * komentarz wywołał dyskusję – ma co najmniej 1 odpowiedź (`reply_count >= 1`),
   * autor recenzji ma reputację większą niż 1 (`user_reputation > 1`).
2. Policz i wypisz, ile recenzji spełnia te kryteria (`.count()`).
3. Wyświetl 10 takich recenzji posortowanych malejąco według liczby odpowiedzi (`reply_count`), pokazując kolumny: `recipe_name`, `user_name`, `stars`, `reply_count` oraz `text`.

In [97]:
# Miejsce na rozwiązanie Ćwiczenia 3:

# 1. Recenzje spełniające jednocześnie trzy warunki
df_low_stars_discussed = df_reviews_clean.filter(
    (F.col("stars").isin(1, 2))
    & (F.col("reply_count") >= 1)
    & (F.col("user_reputation") > 1)
)

# 2. Ile recenzji spełnia kryteria
print("Liczba recenzji:", df_low_stars_discussed.count())

# 3. 10 recenzji z największą liczbą odpowiedzi
df_low_stars_discussed.select(
    "recipe_name", "user_name", "stars", "reply_count", "text"
).orderBy(F.desc("reply_count")).show(10, truncate=40)

Liczba recenzji: 1
+-------------------+-------------+-----+-----------+----------------------------------------+
|        recipe_name|    user_name|stars|reply_count|                                    text|
+-------------------+-------------+-----+-----------+----------------------------------------+
|Winning Apple Crisp|carolanne1210|    1|          1|Sounded interesting, so I gave it a t...|
+-------------------+-------------+-----+-----------+----------------------------------------+



---
## Moduł 4: Statystyki opisowe, grupowanie (`groupBy`), agregacje (`agg`) i tabele krzyżowe

### 4.1. Statystyki opisowe (`describe` vs `summary`)
* `.describe(*cols)` – oblicza podstawowe statystyki (`count`, `mean`, `stddev`, `min`, `max`) dla wybranych kolumn.
* `.summary(*statistics)` – rozszerzona wersja pozwalająca dodatkowo wyznaczyć kwartyle (`25%`, `50%`, `75%`).

In [98]:
# Podsumowanie statystyczne dla kolumn numerycznych związanych z zaangażowaniem:
df_reviews_clean.select(
    "user_reputation", "reply_count", "thumbs_up", "thumbs_down", "best_score", "stars"
).summary("count", "mean", "stddev", "min", "50%", "max").show()

+-------+------------------+--------------------+------------------+------------------+------------------+-----------------+
|summary|   user_reputation|         reply_count|         thumbs_up|       thumbs_down|        best_score|            stars|
+-------+------------------+--------------------+------------------+------------------+------------------+-----------------+
|  count|             18182|               18182|             18182|             18182|             18182|            18182|
|   mean|2.1596084039159607|0.014629853701462985|1.0892641073589264|0.5493345066549334|153.16213837861622| 4.28880211197888|
| stddev|10.014666150794335| 0.13797398993046833|  4.20100357282072| 3.470123983568153|141.07531630977303|1.544786358196531|
|    min|                 0|                   0|                 0|                 0|                 0|                0|
|    50%|                 1|                   0|                 0|                 0|               100|                5|


### 4.2. Grupowanie (`groupBy`) i wielokrotne agregacje (`agg`)
Największą siłą metody `.groupBy(...).agg(...)` w PySparku jest możliwość przekazania wielu funkcji agregujących z modułu `F` jednocześnie oraz nadania wynikowym kolumnom przejrzystych nazw za pomocą `.alias()`.

Sprawdźmy np. 10 przepisów, które zebrały łącznie najwięcej głosów negatywnych (`thumbs_down`), pokazując dla nich jednocześnie łączną liczbę recenzji oraz średnią ocenę:

In [99]:
df_reviews_clean.groupBy("recipe_code", "recipe_name").agg(
    F.count("*").alias("reviews_count"),
    F.sum("thumbs_down").alias("total_thumbs_down"),
    F.sum("thumbs_up").alias("total_thumbs_up"),
    F.round(F.avg("stars"), 2).alias("avg_stars"),
).orderBy(F.desc("total_thumbs_down")).show(10, truncate=30)

+-----------+------------------------------+-------------+-----------------+---------------+---------+
|recipe_code|                   recipe_name|reviews_count|total_thumbs_down|total_thumbs_up|avg_stars|
+-----------+------------------------------+-------------+-----------------+---------------+---------+
|       2832|             Cheeseburger Soup|          725|              488|            734|     4.58|
|       9739|         Best Ever Potato Soup|          161|              354|            622|     4.32|
|      17826|     Amish Breakfast Casserole|          338|              328|            971|     4.53|
|      18345|     Simple Au Gratin Potatoes|          164|              313|            390|     4.48|
|      12003|           Traditional Lasagna|          276|              306|            635|     4.61|
|       4383| First-Place Coconut Macaroons|          145|              301|            483|     3.97|
|      41095|Contest-Winning New England...|          149|              2

### 4.3. Tabele krzyżowe (`stat.crosstab`)
Metoda `df.crosstab(col1, col2)` pozwala błyskawicznie wygenerować macierz częstości (tabelę kontyngencji) dla dwóch zmiennych kategorycznych.

> ⚠️ **Uwaga:** Metoda `crosstab` tworzy tyle kolumn wyjściowych, ile unikalnych wartości znajduje się w drugiej zmiennej (`col2`). Należy jej używać wyłącznie wtedy, gdy druga zmienna ma niewielką liczbę unikalnych kategorii!

In [ ]:
# Rozkład ocen (sentiment_label) w podziale na to, czy pod recenzją pojawiła się odpowiedź (has_replies):
df_reviews_clean.withColumn(
    "has_replies", F.when(F.col("reply_count") > 0, "Tak").otherwise("Nie")
).crosstab("sentiment_label", "has_replies").show()

### Ćwiczenie utrwalające 4 (Grupowanie i rankingi analityczne)
Wykonaj na ramce `df_reviews_clean` poniższe analizy:
1. **Rozkład ocen (`stars`):** Pogrupuj dane po kolumnie `stars`, policz liczbę recenzji dla każdej oceny i posortuj wynik rosnąco według liczby gwiazdek (`0` $\rightarrow$ `5`).
2. **Ranking TOP 10 przepisów wg `best_score`:** Pogrupuj recenzje po `recipe_name` i za pomocą jednego wywołania `.agg(...)` wyznacz dla każdego przepisu:
   * liczbę komentarzy (`comments_count`),
   * sumę wartości w kolumnie `best_score` (`total_best_score`),
   * maksymalną liczbę odpowiedzi pod pojedynczym komentarzem (`max_reply_count`),
   * średnią liczbę gwiazdek zaokrągloną do 2 miejsc po przecinku (`avg_stars`).
   Wyświetl 10 przepisów o najwyższej wartości `total_best_score`.

In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 4:

# 1. Rozkład ocen (stars), rosnąco od 0 do 5
df_reviews_clean.groupBy("stars").agg(
    F.count("*").alias("reviews_count")
).orderBy(F.asc("stars")).show()

# 2. TOP 10 przepisów według sumy best_score
df_reviews_clean.groupBy("recipe_name").agg(
    F.count("*").alias("comments_count"),
    F.sum("best_score").alias("total_best_score"),
    F.max("reply_count").alias("max_reply_count"),
    F.round(F.avg("stars"), 2).alias("avg_stars"),
).orderBy(F.desc("total_best_score")).show(10, truncate=40)

---
## Moduł 5: Wydajność DataFrame, partycjonowanie, plan wykonania (`explain`) i format Parquet

Aby zbadać wpływ typów danych, liczby partycji oraz formatu plików na czas przetwarzania większego wolumenu danych, wygenerujemy syntetyczny zbiór danych pracowników **`employee`**.

> 💡 **Kompatybilność z Lab 7:** W kolejnym laboratorium (`lab_07`) będziemy wykorzystywać plik `employee_1m.csv` (1 milion rekordów) do nauki zapytań Spark SQL, partycjonowania dyskowego oraz wiaderkowania (*bucketing*) w Apache Hive. Poniższy generator został naprawiony (zapisuje poprawnie również ostatni niepełny bufor wierszy) i nie wymaga zewnętrznych zależności.

### 5.1. Generowanie zbioru `employee`

In [ ]:
import random
import time

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

header = ["id", "firstname", "lastname", "age", "salary"]
firstnames = [
    "Adam",
    "Katarzyna",
    "Krzysztof",
    "Marek",
    "Aleksandra",
    "Zbigniew",
    "Wojciech",
    "Mieczysław",
    "Agata",
    "Wisława",
]
lastnames = [
    "Mieczykowski",
    "Kowalski",
    "Malinowski",
    "Szczaw",
    "Glut",
    "Barański",
    "Brzęczyszczykiewicz",
    "Wróblewski",
    "Wlotka",
    "Pysla",
]
age_bounds = {"min": 18, "max": 68}
salary_bounds = {"min": 3200, "max": 12500}


def build_dataset(filename: str, n_rows: int = 1_000_000, chunk_size: int = 200_000):
    """Generuje syntetyczny plik CSV z danymi pracowników (wraz z obsługą końcowego bufora)."""
    if os.path.exists(filename):
        print(f"Plik '{filename}' już istnieje ({os.path.getsize(filename) / (1024**2):.1f} MB). Pomijam generowanie.")
        return

    mu = (salary_bounds["max"] + salary_bounds["min"]) / 2
    sigma = 1000
    rng = random.Random(42)

    with open(filename, "w", encoding="utf-8") as fh:
        fh.write(",".join(header) + "\n")
        buffer = []
        for row_id in tqdm(range(1, n_rows + 1), total=n_rows, desc=f"Generowanie {filename}"):
            buffer.append(
                f"{row_id},{rng.choice(firstnames)},{rng.choice(lastnames)},"
                f"{rng.randint(age_bounds['min'], age_bounds['max'])},"
                f"{rng.normalvariate(mu, sigma):.2f}\n"
            )
            if row_id % chunk_size == 0:
                fh.writelines(buffer)
                buffer.clear()
        # Zapisanie pozostałych wierszy, jeśli n_rows nie jest wielokrotnością chunk_size:
        if buffer:
            fh.writelines(buffer)
            buffer.clear()

    print(f"Zapisano {n_rows:,} wierszy do '{filename}' ({os.path.getsize(filename) / (1024**2):.1f} MB).")

In [ ]:
# Generujemy plik employee_1m.csv (1 mln wierszy, ~35 MB – zajmuje ok. 2-3 sekundy i przyda się również w Lab 7!).
# Możesz zwiększyć parametr n_rows np. do 5_000_000 lub 10_000_000, aby mocniej obciążyć procesor:
EMPLOYEE_CSV = "employee_1m.csv"
build_dataset(EMPLOYEE_CSV, n_rows=1_000_000)

Plik 'employee_1m.csv' już istnieje (36.0 MB). Pomijam generowanie.


### 5.2. Wpływ typów danych (`string` vs `double`) na poprawność i szybkość filtrowania
Wczytajmy wygenerowany plik CSV na dwa sposoby:
1. `df_emp_str` – bez `inferSchema=True` (wszystkie kolumny, w tym `salary` oraz `age`, są łańcuchami znaków `string`),
2. `df_emp_num` – z `inferSchema=True` (`id` i `age` jako `integer`, `salary` jako `double`).

Gdy kolumna `salary` ma typ `string`, czyhają na nas **trzy pułapki**:
1. **Błąd rzutowania ANSI SQL w Spark 4.x (`F.col("salary") > 10000`):** W Sparku 4.x domyślnie włączony jest tryb ANSI (`spark.sql.ansi.enabled = true`). Porównanie kolumny `string` z literałem całkowitym `10000` próbuje rzutować tekst `'7353.16'` na typ całkowity (`BIGINT`), co kończy się błędem `NumberFormatException: [CAST_INVALID_INPUT]`.
2. **Pułapka leksykograficzna (`F.col("salary") > "10000"`):** Porównanie dwóch łańcuchów znaków działa alfabetycznie (znak po znaku), przez co `'9500.00' > '10000'` daje `True` (ponieważ znak `'9'` ma wyższy kod ASCII niż `'1'`)!
3. **Narzut wydajnościowy (`F.col("salary") > 10000.0`):** Porównanie z liczbą zmiennoprzecinkową `10000.0` (lub jawny `.cast("double")`) działa poprawnie, ale zmusza silnik do parsowania tekstu na liczbę zmiennoprzecinkową w locie dla każdego wiersza.

In [ ]:
# 1. Wczytanie jako string (bez inferSchema):
df_emp_str = spark.read.csv(EMPLOYEE_CSV, header=True, inferSchema=False)
print("Schemat df_emp_str:")
df_emp_str.printSchema()

# 2. Wczytanie z wnioskowaniem typów numerycznych:
df_emp_num = spark.read.csv(EMPLOYEE_CSV, header=True, inferSchema=True)
print("Schemat df_emp_num:")
df_emp_num.printSchema()

Schemat df_emp_str:
root
 |-- id: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)

Schemat df_emp_num:
root
 |-- id: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)



In [ ]:
# Sprawdźmy wszystkie 3 zachowania w praktyce:

# A) Pułapka 1: Porównanie kolumny STRING z INT (10000) w trybie ANSI Spark 4.x:
try:
    df_emp_str.filter(F.col("salary") > 10000).count()
except Exception as e:
    print("Pułapka 1 (STRING > INT w Spark 4 ANSI):", str(e).splitlines()[0])

# B) Pułapka 2: Porównanie leksykograficzne (STRING > STRING):
cnt_lex = df_emp_str.filter(F.col("salary") > "10000").count()
print(f"Pułapka 2 (porównanie alfabetyczne STRING > '10000'): {cnt_lex:,} wierszy (BŁĘDNY WYNIK!)")

# C) Porównanie czasu poprawnego filtrowania: rzutowanie w locie (STRING > 10000.0) vs natywny typ DOUBLE:
t0 = time.perf_counter()
cnt_str = df_emp_str.filter(F.col("salary") > 10000.0).count()
time_str = time.perf_counter() - t0

t0 = time.perf_counter()
cnt_num = df_emp_num.filter(F.col("salary") > 10000).count()
time_num = time.perf_counter() - t0

print(f"Wynik poprawny (salary jako string -> cast do double): {cnt_str:,} wierszy | czas: {time_str:.3f} s")
print(f"Wynik poprawny (salary jako natywny double):           {cnt_num:,} wierszy | czas: {time_num:.3f} s")

{"ts": "2026-11-13 12:17:11.627", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '7353.16' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 5 in cell [75]", "line": "", "fragment": "__gt__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1853.count.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '7353.16' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__gt__\" was called from\nline 5 in cell [75]\n\r\n\tat org.apache.spark.sql.errors.QueryE

Pułapka 1 (STRING > INT w Spark 4 ANSI): [CAST_INVALID_INPUT] The value '7353.16' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
Pułapka 2 (porównanie alfabetyczne STRING > '10000'): 1,000,000 wierszy (BŁĘDNY WYNIK!)
Wynik poprawny (salary jako string -> cast do double): 15,670 wierszy | czas: 0.492 s
Wynik poprawny (salary jako natywny double):           15,670 wierszy | czas: 0.406 s


### 5.3. Partycjonowanie (`getNumPartitions`, `repartition`, `coalesce`) oraz plan wykonania (`explain`)
Sprawdźmy na ile partycji została podzielona nasza ramka danych oraz jak wygląda fizyczny plan wykonania zapytania wygenerowany przez optymalizator **Catalyst** (`explain(mode="formatted")`).

In [ ]:
print("Domyślna liczba partycji df_emp_num:", df_emp_num.rdd.getNumPartitions())

# Zmiana liczby partycji za pomocą repartition(4):
df_emp_repart = df_emp_num.repartition(4)
print("Liczba partycji po repartition(4):", df_emp_repart.rdd.getNumPartitions())

Domyślna liczba partycji df_emp_num: 2
Liczba partycji po repartition(4): 4


In [ ]:
# Analiza planu wykonania zapytania (explain) dla filtrowania nazwisk kończących się na 'ski':
query_ski = df_emp_num.filter(F.col("lastname").contains("ski")).select(
    "id", "firstname", "lastname", "salary"
)
query_ski.explain(mode="formatted")

== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [4]: [id#4617, firstname#4618, lastname#4619, salary#4621]
Batched: false
Location: InMemoryFileIndex [file:/c:/Analiza-duzych-zbiorow-danych/lab_06/employee_1m.csv]
PushedFilters: [IsNotNull(lastname), StringContains(lastname,ski)]
ReadSchema: struct<id:int,firstname:string,lastname:string,salary:double>

(2) Filter [codegen id : 1]
Input [4]: [id#4617, firstname#4618, lastname#4619, salary#4621]
Condition : (isnotnull(lastname#4619) AND Contains(lastname#4619, ski))




### 5.4. Zapis i odczyt w kolumnowym formacie Apache Parquet
Format **Apache Parquet** to binarny, kolumnowy format przechowywania danych stworzony dla ekosystemu Big Data:
* Przechowuje w metadanych pełny schemat danych (nie wymaga `inferSchema`),
* Kompresuje dane kolumnami (domyślnie algorytmem **Snappy**), zmniejszając rozmiar na dysku kilkukrotnie względem CSV,
* Wspiera **Column Pruning** (odczytuje z dysku tylko te kolumny, które są wymienione w `select`) oraz **Predicate Pushdown** (pomija całe bloki danych na podstawie statystyk `min`/`max` zapisanych w nagłówkach plików Parquet).

In [ ]:
PARQUET_DIR = os.path.join(DATA_DIR, "employee_parquet")

# Zapisujemy ramkę df_emp_repart (4 partycje) do formatu Parquet:
df_emp_repart.write.mode("overwrite").parquet(PARQUET_DIR)

parquet_files = [f for f in os.listdir(PARQUET_DIR) if f.endswith(".parquet")]
print(f"Liczba utworzonych plików .parquet (odpowiada liczbie partycji): {len(parquet_files)}")
for pf in parquet_files:
    size_mb = os.path.getsize(os.path.join(PARQUET_DIR, pf)) / (1024**2)
    print(f"  - {pf} ({size_mb:.2f} MB)")

Py4JJavaError: An error occurred while calling o1895.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdi

### Ćwiczenie utrwalające 5 (Porównanie wydajności i planu wykonania: CSV vs Parquet)
1. Wczytaj zapisane przed chwilą dane z katalogu `PARQUET_DIR` (`./data/employee_parquet`) do nowej ramki `df_emp_parquet` za pomocą `spark.read.parquet(PARQUET_DIR)`.
2. Zmierz za pomocą `time.perf_counter()` czas wykonania poniższego zapytania analitycznego (filtrowanie + grupowanie + agregacja + akcja `.collect()`) na ramce **CSV (`df_emp_num`)** oraz na ramce **Parquet (`df_emp_parquet`)**:
   * przefiltruj pracowników o pensji `salary > 9500`,
   * pogrupuj po kolumnie `lastname`,
   * oblicz liczbę takich pracowników (`count`) oraz średni wiek (`avg("age")`).
3. Wyświetl i porównaj sformatowany plan wykonania `.explain(mode="formatted")` dla obu ramek. Zwróć uwagę na sekcję `ReadSchema` oraz `PushedFilters` w węźle `Scan csv` vs `Scan parquet`.

In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 5:

---
## 6. Zadanie podsumowujące laboratorium

Wykonaj poniższe dwuczęściowe zadanie podsumowujące, które łączy wszystkie techniki poznane na dzisiejszych zajęciach.

---

### Część A: Dyskretyzacja cech (`Bucketizer`) i analiza grup wiekowych w zbiorze `employee`
W analizie danych często zamieniamy zmienną ciągłą (np. `age`) na przedziały (koszyki / buckety). W PySpark MLlib służy do tego transformator [`pyspark.ml.feature.Bucketizer`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.Bucketizer.html).

> **Przykład użycia `Bucketizer`:**
> ```python
> from pyspark.ml.feature import Bucketizer
>
> # Granice przedziałów domkniętych lewostronnie [a, b):
> splits = [10, 20, 30, 40, 50, 60, 70]
> bucketizer = Bucketizer(splits=splits, inputCol="age", outputCol="age_bucket")
> df_bucketed = bucketizer.transform(df_emp_parquet)
> ```

**Twoje zadanie (Część A):**
1. Wykorzystując `Bucketizer`, podziel kolumnę `age` w ramce `df_emp_parquet` na przedziały co 10 lat: `[10, 20), [20, 30), [30, 40), [40, 50), [50, 60), [60, 70]` i zapisz indeks przedziału w nowej kolumnie `age_bucket`.
2. Dodaj czytelną kolumnę tekstową `age_range` (np. za pomocą `F.when(...)` lub wyrażeń kolumnowych), która zamieni numery bucketów `0.0, 1.0, ..., 5.0` na etykiety tekstowe `"10-19"`, `"20-29"`, `"30-39"`, `"40-49"`, `"50-59"`, `"60-69"`.
3. Wyświetl 15 pierwszych wierszy z nową kolumną, a następnie pogrupuj cały zbiór po `age_bucket` oraz `age_range`, obliczając dla każdej grupy wiekowej:
   * liczbę pracowników (`employees_count`),
   * średnie wynagrodzenie zaokrągloną do 2 miejsc po przecinku (`avg_salary`),
   * minimalne i maksymalne wynagrodzenie (`min_salary`, `max_salary`).
   Posortuj wynik rosnąco według `age_bucket` i wyświetl tabelę wynikową.

---

### Część B: Kompleksowy raport najaktywniejszych recenzentów (*UCI Recipe Reviews*)
Korzystając z oczyszczonej ramki `df_reviews_clean` z pierwszej części laboratorium, przygotuj zestawienie **TOP 20 najbardziej aktywnych użytkowników portalu kulinarnego**:
1. Pogrupuj recenzje po identyfikatorze i nazwie użytkownika (`user_id`, `user_name`).
2. Dla każdego użytkownika oblicz w `.agg(...)`:
   * `reviews_count` – łączną liczbę wystawionych recenzji,
   * `avg_stars` – średnią ocenę gwiazdkową (zaokrągloną do 2 miejsc po przecinku),
   * `avg_review_len` – średnią długość recenzji w znakach (`F.round(F.avg(F.length("text")), 1)`),
   * `total_thumbs_up` – sumę otrzymanych polubień `thumbs_up`,
   * `total_thumbs_down` – sumę otrzymanych głosów negatywnych `thumbs_down`.
3. Przefiltruj wynik tak, aby zostawić tylko recenzentów, którzy napisali **co najmniej 10 recenzji** (`reviews_count >= 10`) oraz których nazwa użytkownika **nie jest** równa `"Anonymous"`.
4. Dodaj kolumnę `approval_rate` (procentowy udział głosów pozytywnych wśród wszystkich głosów oddanych na recenzje danego użytkownika):
   $$\text{approval\_rate} = \text{round}\left(\frac{\text{total\_thumbs\_up}}{\text{total\_thumbs\_up} + \text{total\_thumbs\_down}} \times 100, 2\right)$$
   *(Wskazówka: zabezpiecz się przed dzieleniem przez zero za pomocą `F.when((F.col("total_thumbs_up") + F.col("total_thumbs_down")) > 0, ...).otherwise(0.0)`).*
5. Posortuj użytkowników malejąco według `reviews_count` (a w drugiej kolejności malejąco według `total_thumbs_up`), wybierz 20 pierwszych (`.limit(20)`), wyświetl wynik na ekranie i zapisz go do katalogu `./data/top_20_reviewers.parquet`.

In [ ]:
# Miejsce na rozwiązanie Zadania podsumowującego – Część A (Bucketizer na zbiorze employee):

In [ ]:
# Miejsce na rozwiązanie Zadania podsumowującego – Część B (Raport TOP 20 recenzentów):

In [ ]:
# Po zakończeniu pracy zamykamy sesję Spark:
# spark.stop()